In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Literal, Optional

import numpy as np
from numpy.typing import NDArray
from PIL import Image

# Make an alias for the image format type to save space
ImageF = NDArray[np.float32]  # HxWxC in [0,1]

def load_image_rgb(path: str | Path) -> ImageF:
    """Load an image in RGB from a specified file path."""
    img = Image.open(path).convert("RGB")
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return arr

def save_image_rgb(path: str | Path, img: ImageF) -> None:
    """Save an image in RGB to a specified file path."""
    img8 = np.clip(img * 255.0 + 0.5, 0, 255).astype(np.uint8)
    Image.fromarray(img8, mode="RGB").save(path)

## Pipeline Algorithms
---
### *Algorithm 1: RGB-to-Luma Conversion*  
Compute a weighted sum of the red, green, and blue channels to produce a single-channel image representing perceived brightness, i.e., **luma** ($Y'$). The weights are derived from the ITU-R BT.709 standard, which models human photopic vision sensitivity: $0.2126$ for red, $0.7152$ for green, and $0.0722$ for blue.

Note that this operation is a weighted *channel reduction*, which irreversibly destroys chromatic information. Ergo, reserve this operation for grayscale pipelines.

**Input:** $I \in \mathbb{R}^{H\times W\times 3}$  
**Output:** $Y' \in \mathbb{R}^{H\times W}$

1. $\mathbf{for}\;i \leftarrow1\;\mathbf{to}\; H\; \mathbf{do}$
2. $\quad \mathbf{for}\; j \leftarrow1\;\mathbf{to}\; W\; \mathbf{do}$
3. $\quad\quad R \leftarrow I_{i,j,0}$
4. $\quad\quad G \leftarrow I_{i,j,1}$
5. $\quad\quad B \leftarrow I_{i,j,2}$
6. $\quad\quad Y'_{i,j} \leftarrow 0.2126R + 0.7152G + 0.0722B$
7. $\mathbf{return}\; Y'$
---

### *Algorithm 2: Luma-to-RGB Conversion*  
Convert a single-channel luma image into a 3-channel grayscale RGB image by replicating the luma values across all three color channels (R, G, B). This function primarily exists to make a grayscale image displayable in systems expecting RGB.

Note that this operation is not the inverse of $\mathtt{rgb\_to\_luma}$; chromatic information is permanently lost once it has been discarded.

**Input:** $Y' \in \mathbb{R}^{H\times W}$  
**Output:** $I \in \mathbb{R}^{H\times W\times3}$

1. $\mathbf{for}\;i\leftarrow1\;\mathbf{to}\;H\;\mathbf{do}$
2. $\quad \mathbf{for}\;j\leftarrow1\;\mathbf{to}\;W\;\mathbf{do}$
3. $\quad\quad I_{i,j,0}\leftarrow Y_{i,j}$
4. $\quad\quad I_{i,j,1}\leftarrow Y_{i,j}$
5. $\quad\quad I_{i,j,2}\leftarrow Y_{i,j}$
6. $\mathbf{return}\;I$
---

### *Algorithm 3: sRGB-to-Linear Decoding*  

Convert gamma-encoded sRGB values into linear-light values using the sRGB decoding function. Image arithmetic (blending, filtering, convolution, etc.) should occur in linear space since sRGB encoding is optimized for perceptual uniformity and storage efficiency, not physical accuracy.

**Input:** $X \in \mathbb{R}^{H\times W\times C}$  
**Output:** $L \in \mathbb{R}^{H\times W\times C}$

1. $a\leftarrow0.055$
2. $\mathbf{for}\;\text{each pixel}\;(i,j)\;\text{and channel}\;\mathbf{do}$
3. $\quad x\leftarrow X_{i,j,c}$
4. $\quad \mathbf{if}\;x\leq0.04045\;\mathbf{then}$
5. $\quad\quad L_{i,j,c}\leftarrow x/12.92$
6. $\quad \mathbf{else}$
7. $\quad\quad L_{i,j,c}\leftarrow\left(\frac{x+a}{1+a}\right)^{2.4}$
8. $\mathbf{return}\;L$
---

### *Algorithm 4: Linear-to-sRGB Encoding*  

Convert linear-light values into gamma-encoded sRGB values using the sRGB inverse EOTF (EOTF⁻¹). This is the encoding counterpart to Algorithm 3; apply it as the final step before display or storage, after all linear-space arithmetic is complete.

Note that this operation is the mathematical inverse of $\mathtt{srgb\_to\_linear}$: round-trip fidelity is limited only by floating-point precision. The piecewise structure mirrors Algorithm 3—a linear segment below the threshold $0.0031308$ and a power-curve segment above it—but with exponent $1/2.4$ (compressive) rather than $2.4$ (expansive).

**Input:** $L \in \mathbb{R}^{H\times W\times C}$  
**Output:** $X \in \mathbb{R}^{H\times W\times C}$

1. $a\leftarrow 0.055$
2. $\mathbf{for}\;\text{each pixel}\;(i,j)\;\text{and channel}\;c\;\mathbf{do}$
3. $\quad x\leftarrow L_{i,j,c}$
4. $\quad \mathbf{if}\;x\leq 0.0031308\;\mathbf{then}$
5. $\quad\quad X_{i,j,c}\leftarrow 12.92x$
6. $\quad \mathbf{else}$
7. $\quad\quad X_{i,j,c} \leftarrow(1+a)x^{1/2.4}-a$
8. $\mathbf{return}\;X$
---

In [2]:
def rgb_to_luma(rgb_img: ImageF) -> ImageF:
    """Convert an RGB image to single-channel luma using Rec.709 coefficients."""
    r, g, b = rgb_img[..., 0], rgb_img[..., 1], rgb_img[..., 2]
    luma_img = 0.2126*r + 0.7152*g + 0.0722*b
    return luma_img.astype(np.float32)

def luma_to_grayrgb(luma_img: ImageF) -> ImageF:
    """Convert a single-channel luma image to a 3-channel grayscale RGB image."""
    return np.stack([luma_img, luma_img, luma_img], axis=-1).astype(np.float32)

def srgb_to_linear(srgb: ImageF) -> ImageF:
    """Piecewise implementation of the sRGB EOTF."""
    a = 0.055
    return np.where(
        srgb <= 0.04045,
        srgb / 12.92,
        ((srgb + a) / (1.0 + a)) ** 2.4,
    ).astype(np.float32)

def linear_to_srgb(lin: ImageF) -> ImageF:
    """Piecewise implementation of the sRGB inverse EOTF."""
    a = 0.055
    return np.where(
        lin <= 0.0031308,
        12.92 * lin,
        (1.0 + a) * (lin ** (1 / 2.4)) - a,
    ).astype(np.float32)

### *Algorithm 5: Uniform Level Quantization*

Perform uniform scalar quantization on values in $[0,1]$. Multiply the value by the number of intervals, round to the nearest integer index, then scale back into $[0,1]$.

Given input value $x \in [0,1]$ and number of levels $L$:<br>
1. Verify that $L \geq 2$  
2. Compute the number of intervals: 
$$ 
L - 1 
$$
3. Map the input to index space:
$$
u = x(L - 1)
$$
4. Round to the nearest integer:
$$
k = \mathrm{round}(u)
$$
5. Map back into normalized intensity space:
$$
y = \frac{k}{L - 1}
$$
6. Return $y$  
>For an array, do this elementwise.
---

### *Algorithm 6: Nearest-Neighbor Palette Quantization*

Map the *whole color vector* to the nearest entry in an arbitrary palette. Effectively, find the palette color with the smallest Euclidean distance to the input color, and output that palette color. 

**Input:** $\text{pixel }x \in \mathbb{R}^C, \text{ palette } P={p_1,\dots,p_K}$  
**Output:** $\text{nearest palette color } y$

1. For each $k \in {1, \dots, K}$, compute
$$
D_k \leftarrow \| p_k - x \|_2^2
$$
2. Compute
$$
k^* \leftarrow \arg
$$

In [ ]:
def quantize_levels(pixel: ImageF, levels: int) -> ImageF:
    # pixel: (...) floats in [0,1]
    if levels < 2:
        raise ValueError("levels must be >= 2")
    return np.round(pixel * (levels - 1)) / (levels - 1)

def quantize_palette(pixel: ImageF, palette: ImageF) -> ImageF:
    # pixel: (C,)  palette: (K, C)
    # nearest neighbor in Euclidean RGB
    # TODO: switch to Lab or DKL color space
    diffs = palette - pixel[None, :]
    idx = np.argmin(np.sum(diffs * diffs, axis=1))
    return palette[idx]

